In [ ]:
# === CHALLENGE 2 (TNF-alpha) : PRODUCTION SETUP ===
# Builds the CROPPED target. The smoke test measured 43.3 s per iteration on the full
# two-chain construct (304 + 70 residues) - 7.2x Challenge 1's rate for a 1.7x length
# increase, so the cost scales nearer L^3.7 than L^2. At that rate the whole GPU budget
# buys about six designs, which is not worth running.
#
# Cropping to a 22 A sphere around the switch site keeps all 25 site residues, both
# switch anchors (K166 of chain A, R108 of chain B) and - measured against the intact
# trimer - introduces ZERO artificially exposed surface at the site, fewer than the
# uncropped two-chain construct, because cropping also removes residues that chain C
# had been covering.
import glob, json, os, shutil, subprocess, sys, time
t0 = time.time()
OFF = 76
RADIUS = 22.0


def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print((r.stdout + r.stderr)[-1000:])
        if check:
            raise SystemExit("setup failed")
    return r


print("python", sys.version.split()[0], flush=True)
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip(), flush=True)

os.makedirs("/kaggle/working/params", exist_ok=True)
src = glob.glob("/kaggle/input/**/params/params_model_*.npz", recursive=True)
if src:
    print("reusing %d AF2 param files from the attached input" % len(src), flush=True)
    for f in src:
        dst = "/kaggle/working/params/" + os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy(f, dst)
else:
    print("downloading AF2 params (~3.5 GB) ...", flush=True)
    sh("wget -q -O /kaggle/working/af2.tar "
       "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
    sh("tar -xf /kaggle/working/af2.tar -C /kaggle/working/params")
    sh("rm -f /kaggle/working/af2.tar")
print("AF2 params present:", len(glob.glob("/kaggle/working/params/*.npz")), "files", flush=True)

sh("pip install -q biopython")
import urllib.request
import numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import protein_letters_3to1

os.makedirs("/kaggle/working/targets", exist_ok=True)
raw = "/kaggle/working/1TNF.pdb"
if not os.path.exists(raw):
    open(raw, "wb").write(urllib.request.urlopen(
        "https://files.rcsb.org/download/1TNF.pdb", timeout=180).read())

P = PDBParser(QUIET=True)
mdl = P.get_structure("m", raw)[0]
ca = {}
for ch in mdl:
    for r in ch:
        if r.id[0] == " " and "CA" in r:
            ca[(ch.id, r.id[1])] = r["CA"].get_coord()
centre = (ca[("A", 166 - OFF)] + ca[("B", 108 - OFF)]) / 2
keep = {k for k, v in ca.items() if k[0] in ("A", "B")
        and float(np.linalg.norm(v - centre)) <= RADIUS}


class Crop(Select):
    def accept_chain(self, c): return c.id in ("A", "B")
    def accept_residue(self, r): return r.id[0] == " " and (r.parent.id, r.id[1]) in keep
    def accept_atom(self, a): return a.element != "H"


TARGET = "/kaggle/working/targets/TNF_site_crop.pdb"
st = P.get_structure("t", raw)
io = PDBIO(); io.set_structure(st); io.save(TARGET, Crop())

# The map from construct order to the original chain and UniProt number. AfDesign
# CONCATENATES the target chains into one chain on output and renumbers from 1 - the
# smoke test showed chains {'A': 608, 'B': 140} where three chains were expected - so
# position in this list IS the output residue number minus one. Everything downstream
# maps through this rather than assuming a numbering.
order = []
for ch in P.get_structure("c", TARGET)[0]:
    for r in ch:
        if r.id[0] == " ":
            order.append(dict(chain=ch.id, pdb=r.id[1], uniprot=r.id[1] + OFF,
                              aa=protein_letters_3to1.get(r.get_resname(), "X")))
json.dump(dict(radius=RADIUS, n=len(order), order=order),
          open("/kaggle/working/construct_map.json", "w"))

SITE = [95, 105, 106, 107, 108, 109, 110, 111, 112, 155, 159, 161, 162, 163, 164, 165,
        166, 167, 168, 201, 203, 207, 220, 221, 223]
have = {(o["chain"], o["uniprot"]) for o in order}
print("cropped target: %d residues (A %d, B %d)"
      % (len(order), sum(1 for o in order if o["chain"] == "A"),
         sum(1 for o in order if o["chain"] == "B")), flush=True)
print("site residues retained: %d of %d"
      % (len({o["uniprot"] for o in order} & set(SITE)), len(SITE)), flush=True)
assert ("A", 166) in have and ("B", 108) in have, "a switch anchor was cropped away"
assert len({o["uniprot"] for o in order} & set(SITE)) == len(SITE), "site residue lost"
print("switch anchors present: K166(A), R108(B)", flush=True)

HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"

PROBE = ("python -c \"import jax, jax.lib; assert hasattr(jax.lib,'xla_bridge'); "
         "print(jax.__version__)\"")
r = subprocess.run(PROBE, shell=True, capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip().startswith("0.4.38"):
    print("installing jax 0.4.38 ...", flush=True)
    sh("pip install -q 'jax[cuda12]==0.4.38' 'jaxlib==0.4.38'")
sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")

chk = subprocess.run(
    "cd /kaggle/working && python -c \""
    "import jax; from colabdesign import mk_afdesign_model; "
    "m = mk_afdesign_model(protocol='binder', data_dir='/kaggle/working'); "
    "m.prep_inputs(pdb_filename='%s', chain='A,B', binder_len=65, hotspot='%s'); "
    "print('READY', jax.__version__, jax.devices()); "
    "print('target length seen by the model:', m._target_len)\"" % (TARGET, HOTSPOT),
    shell=True, capture_output=True, text=True)
print(chk.stdout[-500:])
if chk.returncode != 0:
    print("PREP FAILED:"); print(chk.stderr[-1500:])
print("TNF PRODUCTION SETUP DONE in %.1f min" % ((time.time() - t0) / 60))


In [ ]:
# === CHALLENGE 2 (TNF-alpha) : MOUSE TARGET SETUP ===
#
# Objective 2 is mouse cross-reactivity, and it is ranked above affinity. Until now it
# has only been inferred from sequence conservation across each design's human
# epitope. This builds the mouse equivalent of the human target so binding can be
# predicted directly instead.
#
# PDB 2TNF is mouse TNF-alpha at 1.4 A (UniProt P06804, chains A/B/C, a trimer) -
# higher resolution than the human 1TNF used for design.
#
# The numbering is the part that has to be right. Human 1TNF is numbered UniProt-76;
# mouse 2TNF is numbered UniProt-79. Worse, the human->mouse alignment is NOT a
# constant shift: across the 25 site residues the offset is +2 at some positions and
# +3 at others, so there is an indel between the orthologs. Every mouse residue here
# therefore comes from the pairwise alignment, never from arithmetic on a human
# number. On EGFR a three-residue numbering shift silently emptied a result list for
# two rounds, so each step below asserts what it believes.
#
# CORRECTED after the first attempt failed on its own assertion with
# "mouse 168 is GLU, expected LYS". 2TNF's author numbering is NOT a constant offset
# from UniProt: the chain runs 9-157, which is 149 positions, while its DBREF claims
# 1-156, and the observed offset takes BOTH the values 78 and 79 within a single
# chain. Mouse K168 is residue 90, not 89. Residue numbers are therefore derived by
# aligning each observed chain sequence to P06804 and are never computed by adding a
# constant - the same rule already applied to the human construct, which this cell
# originally failed to follow.
import glob, json, os, shutil, subprocess, sys, time
t0 = time.time()
RADIUS = 22.0                  # same sphere as the human crop

# UniProt P06804, mouse TNF-alpha precursor, 235 aa. Embedded so the mapping does not
# depend on a second network fetch at run time.
P06804 = (
    "MSTESMIRDVELAEEALPQKMGGFQNSRRCLCLSLFSFLLVAGATTLFCLLNFGVIGPQR"
    "DEKFPNGLPLISSMAQTLTLRSSSQNSSDKPVAHVVANHQVEEQLEWLSQRANALLANGM"
    "DLKDNQLVVPADGLYLVYSQVLFKGQGCPDYVLLTHTVSRFAISYQEKVNLLSAVKSPCP"
    "KDTPEGAELKPWYEPIYLGGVFQLEKGDQLSAEVNLPKYLDFAESGQVYFGVIAL")

# From the pairwise alignment of P01375 and P06804 (tnf/results/human_to_mouse_index.json).
# Mouse UniProt numbers for the 25 human site residues, in human order.
MOUSE_SITE = [98, 108, 109, 110, 111, 112, 113, 114, 115, 157, 161, 163, 164, 165,
              166, 167, 168, 169, 170, 203, 205, 209, 222, 223, 225]
HUMAN_SITE = [95, 105, 106, 107, 108, 109, 110, 111, 112, 155, 159, 161, 162, 163,
              164, 165, 166, 167, 168, 201, 203, 207, 220, 221, 223]
ANCHOR_K, ANCHOR_R = 168, 111  # mouse equivalents of human K166 and R108


def sh(cmd, check=False):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print("FAILED:", cmd)
        print((r.stdout + r.stderr)[-1000:])
        if check:
            raise SystemExit("setup failed")
    return r


print("python", sys.version.split()[0], flush=True)
print(subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout.strip(), flush=True)

os.makedirs("/kaggle/working/params", exist_ok=True)
src = glob.glob("/kaggle/input/**/params/params_model_*.npz", recursive=True)
if src:
    print("reusing %d AF2 param files from the attached input" % len(src), flush=True)
    for f in src:
        dst = "/kaggle/working/params/" + os.path.basename(f)
        if not os.path.exists(dst):
            shutil.copy(f, dst)
else:
    print("downloading AF2 params (~3.5 GB) ...", flush=True)
    sh("wget -q -O /kaggle/working/af2.tar "
       "https://storage.googleapis.com/alphafold/alphafold_params_2022-12-06.tar")
    sh("tar -xf /kaggle/working/af2.tar -C /kaggle/working/params")
    sh("rm -f /kaggle/working/af2.tar")
print("AF2 params present:", len(glob.glob("/kaggle/working/params/*.npz")), "files", flush=True)

sh("pip install -q biopython")
import urllib.request
import numpy as np
from Bio.PDB import PDBParser, PDBIO, Select
from Bio.PDB.Polypeptide import protein_letters_3to1

os.makedirs("/kaggle/working/targets", exist_ok=True)
raw = "/kaggle/working/2TNF.pdb"
if not os.path.exists(raw):
    open(raw, "wb").write(urllib.request.urlopen(
        "https://files.rcsb.org/download/2TNF.pdb", timeout=180).read())

from Bio import Align
_al = Align.PairwiseAligner()
_al.open_gap_score, _al.extend_gap_score = -11, -1
_al.substitution_matrix = Align.substitution_matrices.load("BLOSUM62")

P = PDBParser(QUIET=True)
mdl = P.get_structure("m", raw)[0]

# residue number -> UniProt index, per chain, by ALIGNMENT not arithmetic
R2U = {}
for cid in ("A", "B"):
    res = [r for r in mdl[cid] if r.id[0] == " "]
    seq = "".join(protein_letters_3to1.get(r.get_resname(), "X") for r in res)
    aln = _al.align(P06804, seq)[0]
    mp = {}
    for (us, ue), (ss, se) in zip(aln.aligned[0], aln.aligned[1]):
        for k in range(int(ue - us)):
            mp[res[int(ss + k)].id[1]] = int(us + k) + 1
    bad = [(n, u) for n, u in mp.items()
           if P06804[u - 1] != protein_letters_3to1.get(mdl[cid][n].get_resname(), "X")]
    assert not bad, "chain %s: %d residues disagree with P06804, e.g. %s" % (cid, len(bad), bad[:3])
    R2U[cid] = mp
    print("chain %s: %d residues mapped to P06804, 0 mismatches, offsets present %s"
          % (cid, len(mp), sorted({u - n for n, u in mp.items()})), flush=True)

U2R = {cid: {u: n for n, u in mp.items()} for cid, mp in R2U.items()}
ca, name3 = {}, {}
for ch in mdl:
    for r in ch:
        if r.id[0] == " " and "CA" in r:
            ca[(ch.id, r.id[1])] = r["CA"].get_coord()
            name3[(ch.id, r.id[1])] = r.get_resname()

assert ANCHOR_K in U2R["A"], "mouse K%d is absent from 2TNF chain A" % ANCHOR_K
assert ANCHOR_R in U2R["B"], "mouse R%d is absent from 2TNF chain B" % ANCHOR_R
kA, rB = ("A", U2R["A"][ANCHOR_K]), ("B", U2R["B"][ANCHOR_R])
assert name3[kA] == "LYS", "mouse %d (resi A%d) is %s, expected LYS" % (ANCHOR_K, kA[1], name3[kA])
assert name3[rB] == "ARG", "mouse %d (resi B%d) is %s, expected ARG" % (ANCHOR_R, rB[1], name3[rB])
print("anchors confirmed in 2TNF: K%d = resi A%d, R%d = resi B%d"
      % (ANCHOR_K, kA[1], ANCHOR_R, rB[1]), flush=True)

centre = (ca[kA] + ca[rB]) / 2
keep = {k for k, v in ca.items() if k[0] in ("A", "B")
        and float(np.linalg.norm(v - centre)) <= RADIUS}


class Crop(Select):
    def accept_chain(self, c): return c.id in ("A", "B")
    def accept_residue(self, r): return r.id[0] == " " and (r.parent.id, r.id[1]) in keep
    def accept_atom(self, a): return a.element != "H"


TARGET = "/kaggle/working/targets/TNF_mouse_site_crop.pdb"
st = P.get_structure("t", raw)
io = PDBIO(); io.set_structure(st); io.save(TARGET, Crop())

order = []
for ch in P.get_structure("c", TARGET)[0]:
    for r in ch:
        if r.id[0] == " ":
            order.append(dict(chain=ch.id, pdb=r.id[1], uniprot=R2U[ch.id][r.id[1]],
                              aa=protein_letters_3to1.get(r.get_resname(), "X")))
H2M = dict(zip(HUMAN_SITE, MOUSE_SITE))
M2H = {m: h for h, m in H2M.items()}
json.dump(dict(species="mouse", pdb="2TNF", radius=RADIUS, n=len(order),
               numbering="per-residue, from alignment to P06804; offset is not constant",
               order=order, mouse_site=MOUSE_SITE, human_site=HUMAN_SITE,
               mouse_to_human_site=M2H),
          open("/kaggle/working/construct_map_mouse.json", "w"))

have = {o["uniprot"] for o in order}
print("cropped mouse target: %d residues (A %d, B %d)"
      % (len(order), sum(1 for o in order if o["chain"] == "A"),
         sum(1 for o in order if o["chain"] == "B")), flush=True)
print("mouse site residues retained: %d of %d" % (len(have & set(MOUSE_SITE)),
                                                  len(MOUSE_SITE)), flush=True)
missing = sorted(set(MOUSE_SITE) - have)
assert not missing, "mouse site residues cropped away: %s" % missing
assert ("A", ANCHOR_K) in {(o["chain"], o["uniprot"]) for o in order}
assert ("B", ANCHOR_R) in {(o["chain"], o["uniprot"]) for o in order}
print("switch anchors present: K%d(A), R%d(B)" % (ANCHOR_K, ANCHOR_R), flush=True)

# Sanity check the alignment itself: every site residue identity must match what the
# human/mouse alignment predicted, or the mapping is wrong somewhere.
seqmap = {o["uniprot"]: o["aa"] for o in order}
HUMAN_AA = dict(zip(HUMAN_SITE, "NLNRRANALTIVSYQTKVNQERFAS"))
bad = [(m, seqmap[m]) for m in MOUSE_SITE if m in seqmap and seqmap[m] not in "ACDEFGHIKLMNPQRSTVWY"]
assert not bad, "unparsable residues %s" % bad
print("mouse site sequence: %s" % "".join(seqmap[m] for m in MOUSE_SITE), flush=True)
print("human site sequence: %s" % "".join(HUMAN_AA[h] for h in HUMAN_SITE), flush=True)

HOTSPOT = ",".join(
    sorted({"%s%d" % (o["chain"], o["pdb"]) for o in order
            if o["uniprot"] in set(MOUSE_SITE)}))
open("/kaggle/working/mouse_hotspot.txt", "w").write(HOTSPOT)
print("mouse hotspot string: %s" % HOTSPOT, flush=True)

PROBE = ("python -c \"import jax, jax.lib; assert hasattr(jax.lib,'xla_bridge'); "
         "print(jax.__version__)\"")
r = subprocess.run(PROBE, shell=True, capture_output=True, text=True)
if r.returncode != 0 or not r.stdout.strip().startswith("0.4.38"):
    print("installing jax 0.4.38 ...", flush=True)
    sh("pip install -q 'jax[cuda12]==0.4.38' 'jaxlib==0.4.38'")
sh("pip install -q git+https://github.com/sokrypton/ColabDesign.git")

chk = subprocess.run(
    "cd /kaggle/working && python -c \""
    "import jax; from colabdesign import mk_afdesign_model; "
    "m = mk_afdesign_model(protocol='binder', data_dir='/kaggle/working'); "
    "m.prep_inputs(pdb_filename='%s', chain='A,B', binder_len=65); "
    "print('READY', jax.__version__, jax.devices()); "
    "print('mouse target length seen by the model:', m._target_len)\"" % TARGET,
    shell=True, capture_output=True, text=True)
print(chk.stdout[-500:])
if chk.returncode != 0:
    print("PREP FAILED:"); print(chk.stderr[-1500:])
print("MOUSE TARGET SETUP DONE in %.1f min" % ((time.time() - t0) / 60))


In [ ]:
# === CHALLENGE 2 (TNF-alpha) : BURIAL AND CHARGE FOR EVERY VERIFIED PLACEMENT ===
#
# The evidence run measured the 12 placements we happened to submit and found that
# burial on binding predicts the charge effect at r = -0.58 while the Cbeta-to-cation
# distance we actually designed with predicts it at r = +0.24. Two submitted designs
# turned out to have ZERO burial, meaning their histidines sit in open solvent when
# bound and there is no mechanism for them to switch at all.
#
# This measures all 35 verified placements across the 14 usable lineages, not just the
# 12 chosen, so the set can be re-ranked on the variable that actually predicts the
# switch rather than the one that does not. No new design work: every sequence here was
# already generated and already passed the both-species test in switch5.
#
# Three defects from the evidence run are fixed here:
#   1. The monomer prediction failed on all 20 designs because protocol="hallucination"
#      needs its own prep_inputs call. Whether our binders fold WITHOUT the target is
#      still unknown, and it is the question the Proteinbase pLDDT numbers raised.
#   2. Cation identity labels read residue numbers straight out of the predicted PDB
#      instead of mapping through the construct map, so the names were in the wrong
#      numbering system. Now mapped, and asserted.
#   3. The run rebuilt its AF2 model 152 times for ~212 predictions, because it
#      alternated human/mouse/monomer per design and each switch forces a rebuild.
#      Work is now done in passes - every human prediction, then every mouse one - so
#      rebuilds drop to roughly one per distinct binder length per species.
#
# Error bars are dropped: the evidence run showed only 2 of 5 parameter sets ever load,
# and we already know the spread is about 0.02 on human and 0.03 on mouse.
import glob, json, os, subprocess, sys, textwrap, time

TIME_BUDGET_MIN = 200

WORKER = r"""
import glob, json, os, sys, time
GPU, SHARD, BUDGET = sys.argv[1], int(sys.argv[2]), float(sys.argv[3])
os.environ["CUDA_VISIBLE_DEVICES"] = GPU
t0 = time.time()
def log(m): print("[w%d] %s [%.1f min]" % (SHARD, m, (time.time()-t0)/60), flush=True)

import numpy as np
from Bio.PDB import PDBParser
from Bio.PDB.SASA import ShrakeRupley
from Bio.PDB.Polypeptide import protein_letters_3to1

HT = "/kaggle/working/targets/TNF_site_crop.pdb"
MT = "/kaggle/working/targets/TNF_mouse_site_crop.pdb"
CMAP = json.load(open("/kaggle/working/construct_map.json"))["order"]
H_HOTSPOT = "A86,A87,A88,A90,A91,A92,B29,B32,B33,B34,B35,B36"
M_HOTSPOT = open("/kaggle/working/mouse_hotspot.txt").read().strip()
CAT_N = {"ARG": ("NE", "NH1", "NH2"), "LYS": ("NZ",)}
MAXS = {"A":129,"R":274,"N":195,"D":193,"C":167,"E":223,"Q":225,"G":104,"H":224,
        "I":197,"L":201,"K":236,"M":224,"F":240,"P":159,"S":155,"T":172,"W":285,
        "Y":263,"V":174}
EPS_EFF, DEBYE = 30.0, 7.9
ANCHORS = {("A", 166), ("B", 108)}   # the conserved, exposed cations we designed against

src = glob.glob("/kaggle/input/**/tnf_switch5.json", recursive=True)
if not src:
    raise SystemExit("tnf_switch5.json not found - attach tnf_switch5_repair_fix")
recs = json.load(open(src[0]))
if len(recs) < 26:
    raise SystemExit("tnf_switch5.json has only %d records, expected 29; it looks "
                     "truncated (Kaggle's preview cuts at 102400 bytes and closes the "
                     "JSON brackets). Refusing to select from it." % len(recs))

jobs = []
for r in recs:
    for v in r.get("variants", []):
        if not v.get("passes"):
            continue
        jobs.append(dict(parent=r["name"], his=v["his_substitutions"], seq=v["sequence"],
                         kind=v["kind"], n_his=v["n_his"], n_anchor=v["n_anchor"],
                         switch_score=v["switch_score"], ref_h=v["human_i_ptm"],
                         ref_m=v["mouse_i_ptm"], ref_mo=len(v.get("mouse_on_site") or []),
                         ctl=r["control"]["sequence"]))
jobs.sort(key=lambda j: (len(j["seq"]), j["parent"], j["his"]))
mine = [j for k, j in enumerate(jobs) if k % 2 == SHARD]
if SHARD == 0:
    log("%d verified placements across %d lineages"
        % (len(jobs), len({j["parent"] for j in jobs})))
log("this worker takes %d placements" % len(mine))

from colabdesign import mk_afdesign_model, clear_mem
_c = {}

def _model(species, L):
    key = (species, L)
    if _c.get("key") != key:
        clear_mem()
        if species == "alone":
            # The evidence run omitted this prep_inputs call and every monomer
            # prediction raised AssertionError: inputs not set {'length': L}.
            af = mk_afdesign_model(protocol="hallucination", data_dir="/kaggle/working")
            af.prep_inputs(length=L)
        else:
            af = mk_afdesign_model(protocol="binder", data_dir="/kaggle/working")
            af.prep_inputs(pdb_filename=HT if species == "human" else MT, chain="A,B",
                           binder_len=L,
                           hotspot=H_HOTSPOT if species == "human" else M_HOTSPOT)
        _c["key"], _c["af"] = key, af
        log("built %s model for length %d" % (species, L))
    return _c["af"]

def predict(seq, species, out_pdb=None):
    af = _model(species, len(seq))
    try:
        af.predict(seq=seq, num_recycles=3, verbose=False)
    except TypeError:
        af.predict(seq=seq, verbose=False)
    lg = af.aux["log"]
    if out_pdb:
        af.save_pdb(out_pdb)
    return (round(float(lg.get("i_ptm", lg.get("ptm", 0.0))), 3),
            round(float(lg.get("plddt", 0.0)), 3))

# ---- burial, with cation names mapped through the construct map ------------------
def dedupe(chain):
    res, seen = [], set()
    for r in list(chain):
        if r.id[0] != " " or r.id[1] in seen:
            chain.detach_child(r.id); continue
        seen.add(r.id[1]); res.append(r)
    return res

def burial(pdb_path, blen, his_idx):
    mdl = PDBParser(QUIET=True).get_structure("c", pdb_path)[0]
    kept = {ch.id: dedupe(ch) for ch in mdl}
    bnd_id = min(kept, key=lambda k: len(kept[k]))
    bres = kept[bnd_id]
    if len(bres) != blen:
        raise ValueError("binder chain %s has %d residues, expected %d"
                         % (bnd_id, len(bres), blen))
    # target residues in file order; colabdesign writes the two target chains merged,
    # so the order must line up with construct_map or every label would be wrong.
    tres = []
    for cid in sorted(k for k in kept if k != bnd_id):
        tres.extend(kept[cid])
    if len(tres) != len(CMAP):
        raise ValueError("target has %d residues, construct map has %d"
                         % (len(tres), len(CMAP)))
    seen_aa = "".join(protein_letters_3to1.get(r.get_resname(), "X") for r in tres)
    want_aa = "".join(o["aa"] for o in CMAP)
    if seen_aa != want_aa:
        raise ValueError("target sequence does not match construct_map")
    cats = []
    for k, r in enumerate(tres):
        for an in CAT_N.get(r.get_resname(), ()):
            if an in r:
                cats.append((CMAP[k]["chain"], CMAP[k]["uniprot"], an,
                             r[an].get_coord()))
    if not cats:
        raise ValueError("no Arg/Lys nitrogens on the target")
    ShrakeRupley().compute(mdl, level="R")
    bound = {}
    for k, r in enumerate(bres):
        aa = protein_letters_3to1.get(r.get_resname())
        if aa: bound[k] = r.sasa / MAXS.get(aa, 200)
    import copy
    solo = copy.deepcopy(mdl[bnd_id])
    sres = [r for r in solo if r.id[0] == " "]
    ShrakeRupley().compute(solo, level="R")
    free = {}
    for k, r in enumerate(sres):
        aa = protein_letters_3to1.get(r.get_resname())
        if aa: free[k] = r.sasa / MAXS.get(aa, 200)
    out = {}
    for k in his_idx:
        r = bres[k]
        best = None
        for an in ("ND1", "NE2"):
            if an not in r: continue
            for cid, unum, catn, xyz in cats:
                d = float(np.linalg.norm(r[an].get_coord() - xyz))
                if best is None or d < best[0]:
                    best = (d, cid, unum, catn)
        ddg = None
        if best is not None:
            ddg = round(332.0 / (EPS_EFF * best[0]) * np.exp(-best[0] / DEBYE), 3)
        out[k] = dict(rel_sasa_free=round(free.get(k, -1), 3),
                      rel_sasa_bound=round(bound.get(k, -1), 3),
                      buried_on_binding=round(free.get(k, 0) - bound.get(k, 0), 3),
                      cation="%s%d%s" % (best[1], best[2], best[3]) if best else None,
                      cation_is_anchor=bool(best and (best[1], best[2]) in ANCHORS),
                      cation_dist=round(best[0], 2) if best else None,
                      clash=bool(best and best[0] < 2.5),
                      screened_coulomb_kcal=ddg)
    return out

os.makedirs("/kaggle/working/bur_pdb", exist_ok=True)

# Build the full prediction list up front, then run it in passes grouped by
# (species, length). The evidence run rebuilt the model 152 times for ~212
# predictions because it interleaved species per design; this does not.
for j in mine:
    j["hidx"] = [int(s[1:-1]) - 1 for s in j["his"].split(",")]
    j["key"] = "%s__%s" % (j["parent"], j["his"].replace(",", "_"))
    j["mut"] = {}
    for pos in j["hidx"]:
        if j["seq"][pos] != "H":
            log("%s expected H at %d, found %s - skipping its probes"
                % (j["key"], pos + 1, j["seq"][pos]))
            continue
        for aa in ("Q", "K"):
            j["mut"][(pos, aa)] = j["seq"][:pos] + aa + j["seq"][pos+1:]

def pass_species(species):
    todo = []
    for j in mine:
        todo.append((j, "base", None, j["seq"]))
        for (pos, aa), s in j["mut"].items():
            todo.append((j, "mut", (pos, aa), s))
    todo.sort(key=lambda x: len(x[3]))
    for j, what, tag, s in todo:
        if (time.time()-t0)/60 > BUDGET:
            log("time budget reached during the %s pass" % species); return
        try:
            out = ("/kaggle/working/bur_pdb/%s.pdb" % j["key"]) if (
                what == "base" and species == "human") else None
            r = predict(s, species, out_pdb=out)
            if what == "base":
                j.setdefault("base", {})[species] = r
            else:
                j.setdefault("probe", {})[(species,) + tag] = r[0]
        except Exception as e:
            j.setdefault("err", []).append("%s %s %s" % (species, what,
                                                         type(e).__name__))
            log("%s %s %s FAILED %s %s" % (j["key"], species, what,
                                           type(e).__name__, str(e)[:100]))

for sp in ("human", "mouse"):
    log("---- %s pass ----" % sp)
    pass_species(sp)

log("---- monomer pass ----")
for j in sorted(mine, key=lambda x: len(x["seq"])):
    if (time.time()-t0)/60 > BUDGET: break
    try:
        j["alone"] = predict(j["seq"], "alone")[1]
    except Exception as e:
        j.setdefault("err", []).append("alone " + type(e).__name__)
        log("%s alone FAILED %s %s" % (j["key"], type(e).__name__, str(e)[:120]))

log("---- burial pass (cpu) ----")
rec_path = "/kaggle/working/tnf_burial_w%d.jsonl" % SHARD
for j in mine:
    rec = dict(parent=j["parent"], his=j["his"], kind=j["kind"], n_his=j["n_his"],
               n_anchor=j["n_anchor"], switch_score=j["switch_score"],
               length=len(j["seq"]), ref_human=j["ref_h"], ref_mouse=j["ref_m"],
               ref_mouse_on_site=j["ref_mo"], errors=j.get("err", []))
    b = j.get("base", {})
    if "human" in b: rec["human_i_ptm"], rec["human_plddt"] = b["human"]
    if "mouse" in b: rec["mouse_i_ptm"], rec["mouse_plddt"] = b["mouse"]
    if "alone" in j:
        rec["alone_plddt"] = j["alone"]
        if "human_plddt" in rec:
            rec["plddt_drop_alone"] = round(rec["human_plddt"] - j["alone"], 3)
    pr = j.get("probe", {})
    chg = {}
    for pos in j["hidx"]:
        q, k = pr.get(("human", pos, "Q")), pr.get(("human", pos, "K"))
        if q is not None and k is not None:
            chg["H%d" % (pos+1)] = dict(gln=q, lys=k, lys_minus_gln=round(k-q, 3))
        qm, km = pr.get(("mouse", pos, "Q")), pr.get(("mouse", pos, "K"))
        if qm is not None and km is not None:
            chg.setdefault("H%d" % (pos+1), {})["mouse_lys_minus_gln"] = round(km-qm, 3)
    if chg: rec["charge"] = chg
    try:
        pdb0 = "/kaggle/working/bur_pdb/%s.pdb" % j["key"]
        rec["burial"] = {("H%d" % (k+1)): v
                         for k, v in burial(pdb0, len(j["seq"]), j["hidx"]).items()}
    except Exception as e:
        rec["burial_error"] = "%s %s" % (type(e).__name__, str(e)[:160])
    bs = [v["buried_on_binding"] for v in (rec.get("burial") or {}).values()]
    rec["min_burial"] = min(bs) if bs else None
    rec["sum_burial"] = round(sum(bs), 3) if bs else None
    cs = [v["lys_minus_gln"] for v in chg.values() if "lys_minus_gln" in v]
    rec["worst_charge"] = max(cs) if cs else None     # least negative = weakest link
    with open(rec_path, "a") as fh:
        fh.write(json.dumps(rec) + chr(10))
    log("%-28s %-12s burial min %-6s sum %-6s charge worst %-7s human %s mouse %s"
        % (j["parent"], j["his"], rec["min_burial"], rec["sum_burial"],
           rec["worst_charge"], rec.get("human_i_ptm"), rec.get("mouse_i_ptm")))
log("worker done")
"""

with open("/kaggle/working/tnf_burial_worker.py", "w") as fh:
    fh.write(textwrap.dedent(WORKER))

subprocess.run("pip install -q biopython", shell=True)
NEEDED = ["/kaggle/working/targets/TNF_site_crop.pdb",
          "/kaggle/working/targets/TNF_mouse_site_crop.pdb",
          "/kaggle/working/construct_map.json",
          "/kaggle/working/mouse_hotspot.txt"]
missing = [p for p in NEEDED if not os.path.exists(p)]
if missing:
    raise SystemExit("setup cells have not run - missing %s" % missing)

procs = []
for shard, gpu in enumerate(["0", "1"]):
    p = subprocess.Popen([sys.executable, "-u", "/kaggle/working/tnf_burial_worker.py",
                          gpu, str(shard), str(TIME_BUDGET_MIN)],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    procs.append(p)
    print("launched burial worker %d on GPU %s" % (shard, gpu), flush=True)

import threading
def pump(p):
    for line in p.stdout:
        print(line.rstrip(), flush=True)
threads = [threading.Thread(target=pump, args=(p,), daemon=True) for p in procs]
[t.start() for t in threads]
for p in procs:
    p.wait()
[t.join(timeout=10) for t in threads]

rows = []
for f in sorted(glob.glob("/kaggle/working/tnf_burial_w*.jsonl")):
    for line in open(f):
        line = line.strip()
        if line:
            rows.append(json.loads(line))
json.dump(rows, open("/kaggle/working/tnf_burial.json", "w"), indent=1)
if not rows:
    raise SystemExit("nothing was measured - read the worker log above. Do NOT read an "
                     "empty table as 'no placement is buried'.")
scored = [r for r in rows if r.get("min_burial") is not None]
if not scored:
    raise SystemExit("no placement produced a burial number - every burial call failed. "
                     "Check burial_error above; this is not a result.")

# rank on the variable that actually predicted the charge effect
scored.sort(key=lambda r: (-(r["min_burial"] or 0), r.get("worst_charge") or 0))
print("\n==== ALL VERIFIED PLACEMENTS, RANKED BY WORST-HISTIDINE BURIAL ====")
print("The evidence run found burial predicts the charge effect at r = -0.58 and the")
print("Cbeta-to-cation distance we designed with predicts it at r = +0.24.")
print("%-14s %-12s %3s %7s %7s %8s %7s %7s %5s"
      % ("lineage", "histidines", "nH", "burial", "sum", "charge", "human", "mouse", "site"))
for r in scored:
    print("%-14s %-12s %3d %7s %7s %8s %7s %7s %5d"
          % (r["parent"], r["his"], r["n_his"], r["min_burial"], r["sum_burial"],
             r["worst_charge"], r.get("human_i_ptm"), r.get("mouse_i_ptm"),
             r["ref_mouse_on_site"]))

dead = [r for r in scored if (r["min_burial"] or 0) < 0.02]
print("\n%d of %d placements have a histidine with essentially no burial (<0.02):"
      % (len(dead), len(scored)))
for r in dead:
    print("  %-14s %-12s  no mechanism for that histidine to switch"
          % (r["parent"], r["his"]))

best = {}
for r in scored:
    if r["parent"] not in best:
        best[r["parent"]] = r
print("\n==== BEST PLACEMENT PER LINEAGE, BY BURIAL ====")
print("%-14s %-12s %3s %7s %8s %7s %7s  %s"
      % ("lineage", "histidines", "nH", "burial", "charge", "human", "mouse", "vs submitted"))
for p, r in sorted(best.items(), key=lambda kv: -(kv[1]["min_burial"] or 0)):
    print("%-14s %-12s %3d %7s %8s %7s %7s"
          % (p, r["his"], r["n_his"], r["min_burial"], r["worst_charge"],
             r.get("human_i_ptm"), r.get("mouse_i_ptm")))

drops = [r["plddt_drop_alone"] for r in rows if "plddt_drop_alone" in r]
if drops:
    drops.sort()
    print("\n==== DOES THE BINDER FOLD WITHOUT THE TARGET? ====")
    print("pLDDT bound minus pLDDT alone, over %d placements: median %+.3f, "
          "range %+.3f to %+.3f" % (len(drops), drops[len(drops)//2], drops[0], drops[-1]))
    print("A large positive number means the binder needs the target to hold its shape.")
else:
    print("\nthe monomer pass produced nothing - whether these fold alone is STILL unknown")

clashes = [(r["parent"], r["his"], k, v["cation_dist"])
           for r in rows for k, v in (r.get("burial") or {}).items() if v.get("clash")]
if clashes:
    print("\n%d histidine(s) sit closer than 2.5 A to a cation, which is a clash rather "
          "than a contact - the predicted structure is wrong there:" % len(clashes))
    for c in clashes:
        print("  %-14s %-12s %s at %.2f A" % c)

anc = sum(1 for r in rows for v in (r.get("burial") or {}).values()
          if v.get("cation_is_anchor"))
tot = sum(1 for r in rows for v in (r.get("burial") or {}).values())
print("\n%d of %d histidines sit nearest to one of the two CONSERVED anchors "
      "(K166A / R108B); the rest face some other cation." % (anc, tot))

print("\n===== COPY FROM HERE =====")
for r in rows:
    print(json.dumps(r, separators=(",", ":")))
print("===== COPY TO HERE =====")
